In [1]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\abdallah\Downloads\My-Github\Green-Innovation-Real-Time-Data-Pipeline\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [3]:
!pip install kafka-python-ng

     ------------------------------------ 232.8/232.8 KB 951.2 kB/s eta 0:00:00


You should consider upgrading via the 'c:\Users\abdallah\Downloads\My-Github\Green-Innovation-Real-Time-Data-Pipeline\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [10]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\abdallah\Downloads\My-Github\Green-Innovation-Real-Time-Data-Pipeline\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [11]:
import os
import requests
import json
import time
from kafka import KafkaProducer
from dotenv import load_dotenv

load_dotenv("../.env")

API_KEY = os.getenv("OPENWEATHER_API_KEY")

try:
    producer = KafkaProducer(
        bootstrap_servers=['localhost:9092'],
        value_serializer=lambda v: json.dumps(v).encode('utf-8')
    )
    print("Kafka Producer initialized successfully!")
except Exception as e:
    print(f"Failed to connect to Kafka: {e}")
    producer = None

def stream_weather_by_coordinates(lat, lon):
    URL = f"http://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"
    
    try:
        response = requests.get(URL)
        data = response.json()

        if response.status_code == 200:
            payload = {
                "city": data.get("name", "Unknown Location"), 
                "latitude": lat,
                "longitude": lon,
                "temp": data['main']['temp'],
                "humidity": data['main']['humidity'],
                "pressure": data['main']['pressure'],
                "sea_level": data['main'].get('sea_level', 0),
                "wind_speed": data['wind']['speed'],
                "clouds": data['clouds']['all'],
                "rain_1h": data.get('rain', {}).get('1h', 0.0),
                "weather_description": data['weather'][0]['description'],
                "timestamp": time.time()
            }

            print(f"Location [{lat}, {lon}] -> City: {payload['city']} | Temp: {payload['temp']}°C | Humidity: {payload['humidity']}%")
            
            if producer:
                producer.send('weather_data', value=payload)
                producer.flush()
                print("Data pushed to Kafka successfully!")
        else:
            print(f"API Error for [{lat}, {lon}]: {data.get('message')}")

    except Exception as e:
        print(f"Unexpected error during streaming: {e}")

test_lat = 28.0871
test_lon = 30.7618

stream_weather_by_coordinates(test_lat, test_lon)

Kafka Producer initialized successfully!
Location [28.0871, 30.7618] -> City: Minya | Temp: 28.17°C | Humidity: 49%
Data pushed to Kafka successfully!
